# Fine-mapping

In [ ]:
[finemap]
parameter: genofile = paths
# pheno_file: give genotype file (in plink)，we can read the gentype matrix. These files are separated by TADs.
parameter: cwd = path("output")
parameter: job_size = 30
parameter: walltime = "50h"
parameter: mem = "30G"
parameter: numThreads = 1
# specify the number of causal variants
# specify the number of traits (phenotypes)
parameter: container = ""
input: genofile, group_by = 1
output: f'{cwd:a}/{_input[0]:bn}.rds'
R: expand = '${ }', stdout = f"{_output:n}.stdout", stderr = f"{_output:n}.stderr", container = container
    library(plink2R)
    library(tidyverse)
    library(MASS)
    library(crayon)
    library(pecotmr)
    library(susieR)
    library(vroom)
    library(pgenlibr)
    library(Rfast)

    # pecotmr GitHub version (conda not yet updated)
    for (f in list.files("/sc/arion/projects/ad-omics/haochen/rss_qc/pecotmr/R/",
                         full.names = TRUE, pattern = "\\.R$")) source(f)
    
    # pecotmr GitHub version (conda not yet updated)
    for (f in list.files("/sc/arion/projects/ad-omics/haochen/rss_qc/susieR/R/",
                         full.names = TRUE, pattern = "\\.R$")) source(f)
    select <- dplyr::select
    
    # Project utilities
    source("/sc/arion/projects/ad-omics/haochen/rss_qc/codes/code/simulate_linreg.R")
    source("/sc/arion/projects/ad-omics/haochen/rss_qc/codes/code/misc.R")
    source("/sc/arion/projects/ad-omics/haochen/rss_qc/codes/code/finemap_qc_ver2.r")
    source_build <- function(dir) {
      env <- new.env(parent = globalenv())
      for (f in list.files(dir, pattern = "\\.R$", full.names = TRUE)) sys.source(f, envir = env)
      env
    }
    susie_hc <- source_build("/sc/arion/projects/ad-omics/haochen/rss_qc/susieR/R/")$susie_rss
    
    # -----------------------------------------------------------------------------
    # Load inputs
    # -----------------------------------------------------------------------------
    path <- ${_input:r}
    dat  <- readRDS(path)
    n    <- dat$n_samples
    
    B_inf   <- 10000   # "large" B so B^-1 is negligible
    B_small <- 489     # finite reference size for the 1000G panel
    B_mix   <- 1e6     # R_finite for eb_mix (per your snippet)
    
    # Safe wrapper: takes the susie fn explicitly, catches errors, returns NULL
    safe_susie <- function(.fn, .label, ...) {
      tryCatch(
        withCallingHandlers(
          .fn(...),
          warning = function(w) invokeRestart("muffleWarning")
        ),
        error = function(e) {
          cat(sprintf("[FAIL] %s: %s\n", .label, conditionMessage(e)))
          NULL
        }
      )
    }
    
    # helper
    
    # ---- in-sample -------------------------------------------------------------
    z  <- dat$insample_input$sumstat$z
    LD <- dat$insample_input$LD
    insample     <- safe_susie(susie_hc, "insample",     z = z, R = LD, n = n)
    insample_SER <- safe_susie(susie_hc, "insample_SER", z = z, R = LD, n = n,
                               L = 1, max_iter = 1)
    insample_mix <- safe_susie(susie_hc, "insample_mix", z = z, R = LD, n = n,
                               R_finite = B_inf, R_mismatch = "eb_mix")
    sum_dt_insample <- data.frame(z = z, n = n, variant = colnames(LD))
    
    
    # ---- ADSP external panel ---------------------------------------------------
    z <- dat$ADSP_input$sumstat$z
    U <- dat$ADSP_input$U
    ADSP     <- safe_susie(susie_hc, "ADSP",     z = z, X = t(U), n = n)
    ADSP_SER <- safe_susie(susie_hc, "ADSP_SER", z = z, X = t(U), n = n,
                           L = 1, max_iter = 1)
    ADSP_mix <- safe_susie(susie_hc, "ADSP_mix", z = z, X = t(U), n = n,
                           R_finite = B_inf, R_mismatch = "eb_mix")
    sum_dt_ADSP <- data.frame(z = z, n = n, variant = colnames(t(U)))
    
    
    # ---- UKB external panel ----------------------------------------------------
    z <- dat$UKB_input$sumstat$z
    U <- dat$UKB_input$U
    UKB     <- safe_susie(susie_hc, "UKB",     z = z, X = t(U), n = n)
    UKB_SER <- safe_susie(susie_hc, "UKB_SER", z = z, X = t(U), n = n,
                          L = 1, max_iter = 1)
    UKB_mix <- safe_susie(susie_hc, "UKB_mix", z = z, X = t(U), n = n,
                          R_finite = B_inf, R_mismatch = "eb_mix")
    sum_dt_UKB <- data.frame(z = z, n = n, variant = colnames(t(U)))
    
    
    # ---- 1000G external panel --------------------------------------------------
    z  <- dat$kg_input$sumstat$z
    LD <- dat$kg_input$LD
    kg     <- safe_susie(susie_hc, "kg",     z = z, R = LD, n = n)
    kg_SER <- safe_susie(susie_hc, "kg_SER", z = z, R = LD, n = n,
                         L = 1, max_iter = 1)
    kg_mix <- safe_susie(susie_hc, "kg_mix", z = z, R = LD, n = n,
                         R_finite = B_small, R_mismatch = "eb_mix")
    sum_dt_kg <- data.frame(z = z, n = n, variant = colnames(LD))
    



    # -----------------------------------------------------------------------------
    # Collect fits
    # -----------------------------------------------------------------------------

    result_list <- list(
      insample                      = insample,
      insample_SER                  = insample_SER,
      insample_mix                  = insample_mix,
    
      ADSP                      = ADSP,
      ADSP_SER                  = ADSP_SER,
      ADSP_mix                  = ADSP_mix,
    
      UKB                      = UKB,
      UKB_SER                  = UKB_SER,
      UKB_mix                  = UKB_mix,
    
      kg                      = kg,
      kg_SER                  = kg_SER,
      kg_mix                  = kg_mix
    )

    output <- list(
      result_list = result_list,
      n_sample    = n,
      input_path  = path,
      true_vars   = dat$true_vars
    )
    saveRDS(output, ${_output:r}) 

In [ ]:
# ===== Base config =====
base_dir="/sc/arion/projects/ad-omics/haochen/rss_qc/2026_output/"
job_name="Gtex_fine"
job_dir="${base_dir}/${job_name}"

# ===== Create structure =====
mkdir -p "${job_dir}"/{code,result,log}
cd "${job_dir}/code"

# ===== Base SLURM script template (named as job_name) =====
cat << 'EOF' > ${job_name}.sh
#!/bin/bash -l
#BSUB -P acc_ad-omics
#BSUB -q premium
#BSUB -J JOB_NAME
#BSUB -n 5
#BSUB -R span[hosts=1]
#BSUB -R rusage[mem=30000]
#BSUB -W 72:00
#BSUB -o /WORK_DIR/log/JOB_NAME.%J.out
#BSUB -e /WORK_DIR/log/JOB_NAME.%J.err

export PATH=/sc/arion/projects/ad-omics/haochen/env/R_sc/bin:$PATH:/usr/bin:/bin

sos run "/sc/arion/projects/ad-omics/haochen/rss_qc/2026_Simulation_Sinai/09.Fig2F_3_S5_S6.Finemapping.ipynb" finemap \
    --genofile `ls /sc/arion/projects/ad-omics/haochen/rss_qc/2026_output/Gtex_simulation/result/h2g_H2G/causal_CAUSAL/*.rds` \
    --cwd WORK_DIR/result/h2g_H2G/causal_CAUSAL
EOF

chmod +x ${job_name}.sh

base_sh=${job_name}.sh
work_dir="${job_dir}"

for causal in 1 2 3; do
  for h2g in 0.02 0.03 0.05 0.1 0.15; do
      output_script="${job_name}_causal_${causal}_h2g_${h2g}.sh"
      sed -e "s|CAUSAL|${causal}|g" \
          -e "s|WORK_DIR|${work_dir}|g" \
          -e "s|H2G|${h2g}|g" \
          -e "s|JOB_NAME|${job_name}|g" \
        "${base_sh}" > "${output_script}"

        sb "${output_script}"   # use sb if you aliased
  done
done

In [ ]:
# ===== Base config =====
base_dir="/sc/arion/projects/ad-omics/haochen/rss_qc/2026_output/"
job_name="Gtex_EUR_fine"
job_dir="${base_dir}/${job_name}"

# ===== Create structure =====
mkdir -p "${job_dir}"/{code,result,log}
cd "${job_dir}/code"

# ===== Base SLURM script template (named as job_name) =====
cat << 'EOF' > ${job_name}.sh
#!/bin/bash -l
#BSUB -P acc_ad-omics
#BSUB -q premium
#BSUB -J JOB_NAME
#BSUB -n 5
#BSUB -R span[hosts=1]
#BSUB -R rusage[mem=30000]
#BSUB -W 72:00
#BSUB -o /WORK_DIR/log/JOB_NAME.%J.out
#BSUB -e /WORK_DIR/log/JOB_NAME.%J.err

export PATH=/sc/arion/projects/ad-omics/haochen/env/R_sc/bin:$PATH:/usr/bin:/bin

sos run "/sc/arion/projects/ad-omics/haochen/rss_qc/2026_Simulation_Sinai/09.Fig2F_3_S5_S6.Finemapping.ipynb" finemap \
    --genofile `ls /sc/arion/projects/ad-omics/haochen/rss_qc/2026_output/Gtex_EUR_simulation/result/h2g_H2G/causal_CAUSAL/*.rds` \
    --cwd WORK_DIR/result/h2g_H2G/causal_CAUSAL
EOF

chmod +x ${job_name}.sh

base_sh=${job_name}.sh
work_dir="${job_dir}"

for causal in 1 2 3; do
  for h2g in 0.02 0.03 0.05 0.1 0.15; do
      output_script="${job_name}_causal_${causal}_h2g_${h2g}.sh"
      sed -e "s|CAUSAL|${causal}|g" \
          -e "s|WORK_DIR|${work_dir}|g" \
          -e "s|H2G|${h2g}|g" \
          -e "s|JOB_NAME|${job_name}|g" \
        "${base_sh}" > "${output_script}"

        sb "${output_script}"   # use sb if you aliased
  done
done

In [ ]:
# ===== Base config =====
base_dir="/sc/arion/projects/ad-omics/haochen/rss_qc/2026_output/"
job_name="Hetero_fine"
job_dir="${base_dir}/${job_name}"

# ===== Create structure =====
mkdir -p "${job_dir}"/{code,result,log}
cd "${job_dir}/code"

# ===== Base SLURM script template (named as job_name) =====
cat << 'EOF' > ${job_name}.sh
#!/bin/bash -l
#BSUB -P acc_ad-omics
#BSUB -q premium
#BSUB -J JOB_NAME
#BSUB -n 3
#BSUB -R span[hosts=1]
#BSUB -R rusage[mem=30000]
#BSUB -W 72:00
#BSUB -o /WORK_DIR/log/JOB_NAME.%J.out
#BSUB -e /WORK_DIR/log/JOB_NAME.%J.err

export PATH=/sc/arion/projects/ad-omics/haochen/env/R_sc/bin:$PATH:/usr/bin:/bin

sos run "/sc/arion/projects/ad-omics/haochen/rss_qc/2026_Simulation_Sinai/09.Fig2F_3_S5_S6.Finemapping.ipynb" finemap \
    --genofile `ls /sc/arion/projects/ad-omics/haochen/rss_qc/2026_output/Hetero_sim/result/h2g_H2G/causal_CAUSAL/*.rds` \
    --cwd WORK_DIR/result/h2g_H2G/causal_CAUSAL
EOF

chmod +x ${job_name}.sh

base_sh=${job_name}.sh
work_dir="${job_dir}"

for causal in 2 3; do
  for h2g in 0.05 0.075 0.10 0.15; do
      output_script="${job_name}_causal_${causal}_h2g_${h2g}.sh"
      sed -e "s|CAUSAL|${causal}|g" \
          -e "s|WORK_DIR|${work_dir}|g" \
          -e "s|H2G|${h2g}|g" \
          -e "s|JOB_NAME|${job_name}|g" \
        "${base_sh}" > "${output_script}"

        sb "${output_script}"   # use sb if you aliased
  done
done

In [ ]:
[finemap_GWAS]
parameter: genofile = paths
# pheno_file: give genotype file (in plink)，we can read the gentype matrix. These files are separated by TADs.
parameter: cwd = path("output")
parameter: job_size = 30
parameter: walltime = "50h"
parameter: mem = "30G"
parameter: numThreads = 1
# specify the number of causal variants
# specify the number of traits (phenotypes)
parameter: container = ""
input: genofile, group_by = 1
output: f'{cwd:a}/{_input[0]:bn}.rds'
R: expand = '${ }', stdout = f"{_output:n}.stdout", stderr = f"{_output:n}.stderr", container = container
    library(plink2R)
    library(tidyverse)
    library(MASS)
    library(crayon)
    library(pecotmr)
    library(susieR)
    library(vroom)
    library(pgenlibr)
    library(Rfast)

    # pecotmr GitHub version (conda not yet updated)
    for (f in list.files("/sc/arion/projects/ad-omics/haochen/rss_qc/pecotmr/R/",
                         full.names = TRUE, pattern = "\\.R$")) source(f)
    
    # pecotmr GitHub version (conda not yet updated)
    for (f in list.files("/sc/arion/projects/ad-omics/haochen/rss_qc/susieR/R/",
                         full.names = TRUE, pattern = "\\.R$")) source(f)
    select <- dplyr::select
    
    # Project utilities
    source("/sc/arion/projects/ad-omics/haochen/rss_qc/codes/code/simulate_linreg.R")
    source("/sc/arion/projects/ad-omics/haochen/rss_qc/codes/code/misc.R")
    source("/sc/arion/projects/ad-omics/haochen/rss_qc/codes/code/finemap_qc_ver2.r")
    
    # -----------------------------------------------------------------------------
    # Two susieR builds, each in its own environment so they never clobber.
    # Each susie_*() carries its enclosing env, so internal helper lookups resolve
    # against the build it came from.
    #   susie_hc : Haochen multi-panel / R_finite build -> vanilla + _pop (eb)
    #   susie_xw : Xuewei ld_mismatch build             -> _mix  (eb_mix)
    # -----------------------------------------------------------------------------
    source_build <- function(dir) {
      env <- new.env(parent = globalenv())
      for (f in list.files(dir, pattern = "\\.R$", full.names = TRUE)) sys.source(f, envir = env)
      env
    }
    susie_hc <- source_build("/sc/arion/projects/ad-omics/haochen/rss_qc/susieR/R/")$susie_rss
    susie_xw <- source_build("/sc/arion/projects/ad-omics/haochen/rss_qc/codes/susieR_July17_2026_singleL/")$susie_rss
    colocboost_xw <- source_build("/sc/arion/projects/ad-omics/haochen/rss_qc/codes/colocboost_July22_2026")$colocboost
    
    # -----------------------------------------------------------------------------
    # Load inputs
    #   dat  : precomputed R matrices for gtex (in-sample), ADSP, UKB
    #   dat2 : actual stochastic sketches U (p x B) for ADSP, UKB, used by the
    #          R_finite = TRUE path and the multi-panel mixture
    # -----------------------------------------------------------------------------
    path <- ${_input:r}
    dat  <- readRDS(path)
    n    <- dat$n_samples

    # Effective reference sizes
    B_inf  <- 10000      # "large" B so that B^-1 is negligible
    B_small = 489

    safe_susie <- function(.fun, .label, ...) {
      tryCatch(
        withCallingHandlers(
          .fun(...),
          warning = function(w) {
            invokeRestart("muffleWarning")
          }
        ),
        error = function(e) {
          cat(sprintf("[FAIL] %s: %s\n", .label, conditionMessage(e)))
          NULL
        }
      )
    }

    # ---- ADSP external panel ---------------------------------------------------
    z <- dat$ADSP_input$sumstat$z
    U <- dat$ADSP_input$U
    ADSP     <- safe_susie(susie_hc, "ADSP",     z = z, X = t(U), n = n)
    ADSP_SER <- safe_susie(susie_hc, "ADSP_SER", z = z, X = t(U), n = n,
                           L = 1, max_iter = 1)
    ADSP_mix <- safe_susie(susie_hc, "ADSP_mix", z = z, X = t(U), n = n,
                           R_finite = B_inf, R_mismatch = "eb_mix")
    
    # ---- UKB external panel ----------------------------------------------------
    z <- dat$UKB_input$sumstat$z
    U <- dat$UKB_input$U
    UKB     <- safe_susie(susie_hc, "UKB",     z = z, X = t(U), n = n)
    UKB_SER <- safe_susie(susie_hc, "UKB_SER", z = z, X = t(U), n = n,
                          L = 1, max_iter = 1)
    UKB_mix <- safe_susie(susie_hc, "UKB_mix", z = z, X = t(U), n = n,
                          R_finite = B_inf, R_mismatch = "eb_mix")
    
    # ---- 1000G external panel --------------------------------------------------
    z  <- dat$kg_input$sumstat$z
    LD <- dat$kg_input$LD
    kg     <- safe_susie(susie_hc, "kg",     z = z, R = LD, n = n)
    kg_SER <- safe_susie(susie_hc, "kg_SER", z = z, R = LD, n = n,
                         L = 1, max_iter = 1)
    kg_mix <- safe_susie(susie_hc, "kg_mix", z = z, R = LD, n = n,
                         R_finite = B_small, R_mismatch = "eb_mix")


    # -----------------------------------------------------------------------------
    # Collect fits
    # -----------------------------------------------------------------------------

    result_list <- list(
      ADSP                      = ADSP,
      ADSP_SER                  = ADSP_SER,
      ADSP_mix                  = ADSP_mix,
    
      UKB                      = UKB,
      UKB_SER                  = UKB_SER,
      UKB_mix                  = UKB_mix,
    
      kg                      = kg,
      kg_SER                  = kg_SER,
      kg_mix                  = kg_mix
    )

    output <- list(
      result_list = result_list,
      n_sample    = n,
      input_path  = path,
      true_vars   = dat$true_vars
    )
    
     saveRDS(output, ${_output:r})

In [ ]:
# ===== Base config =====
base_dir="/sc/arion/projects/ad-omics/haochen/rss_qc/2026_output/"
job_name="GWAS_fine"
job_dir="${base_dir}/${job_name}"

# ===== Create structure =====
mkdir -p "${job_dir}"/{code,result,log}
cd "${job_dir}/code"

# ===== Base SLURM script template (named as job_name) =====
cat << 'EOF' > ${job_name}.sh
#!/bin/bash -l
#BSUB -P acc_ad-omics
#BSUB -q premium
#BSUB -J JOB_NAME
#BSUB -n 3
#BSUB -R span[hosts=1]
#BSUB -R rusage[mem=30000]
#BSUB -W 42:00
#BSUB -o /WORK_DIR/log/JOB_NAME.%J.out
#BSUB -e /WORK_DIR/log/JOB_NAME.%J.err

export PATH=/sc/arion/projects/ad-omics/haochen/env/R_sc/bin:$PATH:/usr/bin:/bin

sos run "/sc/arion/projects/ad-omics/haochen/rss_qc/2026_Simulation_Sinai/09.Fig2F_3_S5_S6.Finemapping.ipynb" finemap_GWAS \
    --genofile `ls /sc/arion/projects/ad-omics/haochen/rss_qc/2026_output/GWAS_sim/result/h2g_H2G/causal_CAUSAL/*.rds` \
    --cwd WORK_DIR/result/h2g_H2G/causal_CAUSAL
EOF

chmod +x ${job_name}.sh

base_sh=${job_name}.sh
work_dir="${job_dir}"

for causal in 1 2 3; do
    for h2g in 0.005 0.01; do
      output_script="${job_name}_causal_${causal}_h2g_${h2g}.sh"
      sed -e "s|CAUSAL|${causal}|g" \
          -e "s|WORK_DIR|${work_dir}|g" \
          -e "s|H2G|${h2g}|g" \
          -e "s|JOB_NAME|${job_name}|g" \
        "${base_sh}" > "${output_script}"

        sb "${output_script}"   # use sb if you aliased
  done
done

In [3]:
[LD_mismatch]
parameter: resfile = paths
# All region .rds files for one setting; combined into ONE output table.
parameter: cwd = path("output")
parameter: job_size = 10
parameter: walltime = "100h"
parameter: mem = "72G"
parameter: numThreads = 3
parameter: container = ""
# ---- Metric knobs ------------------------------------------------------------
parameter: ld_thresh = 0.05    # keep off-diagonal pairs with |R_insample| > this
parameter: min_pairs = 10      # skip references with fewer kept pairs
# Variants guaranteed identical/in order across objects. Allele flip = reference
# column name differs from insample at that position -> negate that variant.
input: resfile
output: f'{cwd:a}/ld_mismatch_combined.rds', f'{cwd:a}/ld_mismatch_combined.csv'
task: trunk_workers = 1, walltime = walltime, mem = mem, cores = numThreads,
      tags = f'{step_name}_{_output[0]:bn}'
R:  expand = '${ }', stdout = f"{_output[0]:n}.stdout", stderr = f"{_output[0]:n}.stderr", container = container

    suppressMessages(library(parallel))
    files <- c(${_input:r,})

    cov2cor_safe <- function(M) {
      d <- diag(M); if (any(!is.finite(d)) || any(d <= 0)) return(M); cov2cor(M)
    }
    ld_ids <- function(M) { id <- rownames(M); if (is.null(id)) id <- colnames(M); id }

    ## per-region worker: returns a 3-row data.frame (ADSP/UKB/KG), or NULL on error
    process_one <- function(f) {
      out <- tryCatch({
        res <- readRDS(f)
        insample_LD <- res$insample_input$LD
        ADSP_U <- res$ADSP_input$U
        UKB_U  <- res$UKB_input$U
        KG_LD  <- res$kg_input$LD

        R_ins   <- cov2cor_safe(insample_LD)
        ins_ids <- ld_ids(insample_LD)
        p       <- nrow(R_ins)

        flip_sign <- function(ref_ids) ifelse(ref_ids == ins_ids, 1, -1)

        R_from_dense <- function(ref_LD) {
          s <- flip_sign(ld_ids(ref_LD))
          list(R = cov2cor_safe(outer(s, s) * ref_LD), B = NA_real_, n_flip = sum(s < 0))
        }
        R_from_sketch <- function(U) {
          if (nrow(U) != p) U <- t(U)
          s  <- flip_sign(rownames(U))
          Uf <- U * s
          list(R = cov2cor(tcrossprod(Uf) / ncol(Uf)), B = ncol(Uf), n_flip = sum(s < 0))
        }
        metric <- function(h) {
          ut  <- upper.tri(R_ins)
          ins <- R_ins[ut]; ref <- h$R[ut]
          ok  <- is.finite(ins) & is.finite(ref) & (abs(ins) > ${ld_thresh})
          if (sum(ok) < ${min_pairs})
            return(c(mse = NA_real_, med_sq = NA_real_, max_abs_diff = NA_real_,
                     concord = NA_real_, D2 = NA_real_, n_pairs = sum(ok)))
          d    <- ref[ok] - ins[ok]
          c(mse          = mean(d^2),
            med_sq       = median(d^2),
            max_abs_diff = max(abs(d)),
            concord      = suppressWarnings(cor(ref[ok], ins[ok])),
            D2           = if (is.finite(h$B)) mean(d^2) - mean((1 + ref[ok]^2) / h$B) else NA_real_,
            n_pairs      = sum(ok))
        }

        H <- list(ADSP = R_from_sketch(ADSP_U),
                  UKB  = R_from_sketch(UKB_U),
                  KG   = R_from_dense(KG_LD))
        m <- t(sapply(H, metric))
        data.frame(file = basename(f), reference = names(H), m,
                   p = p, B = sapply(H, `[[`, "B"), n_flip = sapply(H, `[[`, "n_flip"),
                   row.names = NULL, check.names = FALSE)
      }, error = function(e) {
        message("FAILED: ", basename(f), " : ", conditionMessage(e)); NULL
      })
      out
    }

    parts <- mclapply(files, process_one, mc.cores = ${numThreads})
    ok    <- !vapply(parts, is.null, logical(1))
    dat   <- do.call(rbind, parts[ok])

    saveRDS(dat, ${_output[0]:r})
    write.csv(dat, ${_output[1]:r}, row.names = FALSE)
    cat(sprintf("combined %d/%d regions -> %d rows\n", sum(ok), length(files), nrow(dat)))

In [ ]:
# ===== Base config =====
base_dir="/sc/arion/projects/ad-omics/haochen/rss_qc/2026_output/"
job_name="LD_mismatch"
job_dir="${base_dir}/${job_name}"
sim_result="/sc/arion/projects/ad-omics/haochen/rss_qc/2026_output/Gtex_simulation/result/h2g_0.1/causal_1/"
# ===== Create structure =====
mkdir -p "${job_dir}"/{code,result,log}
cd "${job_dir}/code"

# ===== Base LSF script template (named as job_name) =====
cat << 'EOF' > ${job_name}.sh
#!/bin/bash -l
#BSUB -P acc_ad-omics
#BSUB -q premium
#BSUB -J JOB_NAME
#BSUB -n 2
#BSUB -R span[hosts=1]
#BSUB -R rusage[mem=30000]
#BSUB -W 42:00
#BSUB -o /WORK_DIR/log/JOB_NAME.%J.out
#BSUB -e /WORK_DIR/log/JOB_NAME.%J.err

export PATH=/sc/arion/projects/ad-omics/haochen/env/R_sc/bin:$PATH:/usr/bin:/bin

sos run "/sc/arion/projects/ad-omics/haochen/rss_qc/2026_Simulation_Sinai/09.Fig2F_3_S5_S6.Finemapping.ipynb" LD_mismatch \
    --resfile `ls SIM_RESULT/*.rds` \
    --cwd WORK_DIR/
EOF
chmod +x ${job_name}.sh
base_sh=${job_name}.sh
work_dir="${job_dir}"

output_script="${job_name}_submit.sh"
sed -e "s|WORK_DIR|${work_dir}|g" \
    -e "s|JOB_NAME|${job_name}|g" \
    -e "s|SOS_FILE|${sos_file}|g" \
    -e "s|SIM_RESULT|${sim_result}|g" \
  "${base_sh}" > "${output_script}"
sb ${output_script}   # uncomment to submit

In [ ]:
[LD_pairwise]
parameter: resfile = paths
parameter: cwd = path("output")
parameter: job_size = 10
parameter: walltime = "100h"
parameter: mem = "72G"
parameter: numThreads = 3
parameter: container = ""
parameter: ld_thresh = 0.05    # keep off-diagonal pairs with max(|r_a|,|r_b|) > this
parameter: min_pairs = 10
input: resfile
output: f'{cwd:a}/ld_pairwise_combined.rds', f'{cwd:a}/ld_pairwise_combined.csv'
task: trunk_workers = 1, walltime = walltime, mem = mem, cores = numThreads,
      tags = f'{step_name}_{_output[0]:bn}'
R:  expand = '${ }', stdout = f"{_output[0]:n}.stdout", stderr = f"{_output[0]:n}.stderr", container = container

    suppressMessages(library(parallel))
    files <- c(${_input:r,})

    cov2cor_safe <- function(M) {
      d <- diag(M); if (any(!is.finite(d)) || any(d <= 0)) return(M); cov2cor(M)
    }
    ld_ids <- function(M) { id <- rownames(M); if (is.null(id)) id <- colnames(M); id }

    ## per-region worker: 3 pairwise rows (ADSP_vs_UKB / ADSP_vs_KG / UKB_vs_KG)
    process_one <- function(f) {
      tryCatch({
        res <- readRDS(f)
        insample_LD <- res$insample_input$LD   # used ONLY as allele-orientation anchor
        ADSP_U <- res$ADSP_input$U
        UKB_U  <- res$UKB_input$U
        KG_LD  <- res$kg_input$LD

        ins_ids <- ld_ids(insample_LD)
        p       <- nrow(insample_LD)

        flip_sign <- function(ref_ids) ifelse(ref_ids == ins_ids, 1, -1)

        R_from_dense <- function(ref_LD) {
          s <- flip_sign(ld_ids(ref_LD))
          list(R = cov2cor_safe(outer(s, s) * ref_LD), B = NA_real_, n_flip = sum(s < 0))
        }
        R_from_sketch <- function(U) {
          if (nrow(U) != p) U <- t(U)
          s  <- flip_sign(rownames(U))
          Uf <- U * s
          list(R = cov2cor(tcrossprod(Uf) / ncol(Uf)), B = ncol(Uf), n_flip = sum(s < 0))
        }

        ## pairwise metric on two aligned panels
        metric <- function(ha, hb) {
          ut <- upper.tri(ha$R)
          a  <- ha$R[ut]; b <- hb$R[ut]
          ok <- is.finite(a) & is.finite(b) & (pmax(abs(a), abs(b)) > ${ld_thresh})
          if (sum(ok) < ${min_pairs})
            return(c(mse = NA_real_, med_sq = NA_real_, max_abs_diff = NA_real_,
                     concord = NA_real_, D2 = NA_real_, n_pairs = sum(ok)))
          d     <- a[ok] - b[ok]
          noise <- rep(0, sum(ok))                       # subtract each sketch's sampling variance
          if (is.finite(ha$B)) noise <- noise + (1 + a[ok]^2) / ha$B
          if (is.finite(hb$B)) noise <- noise + (1 + b[ok]^2) / hb$B
          c(mse          = mean(d^2),
            med_sq       = median(d^2),
            max_abs_diff = max(abs(d)),
            concord      = suppressWarnings(cor(a[ok], b[ok])),
            D2           = mean(d^2) - mean(noise),
            n_pairs      = sum(ok))
        }

        H <- list(InSample = list(R = cov2cor_safe(insample_LD), B = NA_real_, n_flip = 0L),
                  ADSP     = R_from_sketch(ADSP_U),
                  UKB      = R_from_sketch(UKB_U),
                  KG       = R_from_dense(KG_LD))

        combos <- combn(names(H), 2, simplify = FALSE)
        do.call(rbind, lapply(combos, function(cc) {
          ha <- H[[cc[1]]]; hb <- H[[cc[2]]]
          data.frame(file = basename(f), comparison = paste(cc[1], cc[2], sep = "_vs_"),
                     t(metric(ha, hb)), p = p,
                     B_a = ha$B, B_b = hb$B, n_flip_a = ha$n_flip, n_flip_b = hb$n_flip,
                     row.names = NULL, check.names = FALSE)
        }))
      }, error = function(e) { message("FAILED: ", basename(f), " : ", conditionMessage(e)); NULL })
    }

    parts <- mclapply(files, process_one, mc.cores = ${numThreads})
    ok    <- !vapply(parts, is.null, logical(1))
    dat   <- do.call(rbind, parts[ok])

    saveRDS(dat, ${_output[0]:r})
    write.csv(dat, ${_output[1]:r}, row.names = FALSE)
    cat(sprintf("combined %d/%d regions -> %d rows\n", sum(ok), length(files), nrow(dat)))

In [ ]:
# ===== Base config =====
base_dir="/sc/arion/projects/ad-omics/haochen/rss_qc/2026_output/"
job_name="LD_pairwise"
job_dir="${base_dir}/${job_name}"
sim_result="/sc/arion/projects/ad-omics/haochen/rss_qc/2026_output/Gtex_simulation/result/h2g_0.1/causal_1/"
# ===== Create structure =====
mkdir -p "${job_dir}"/{code,result,log}
cd "${job_dir}/code"

# ===== Base LSF script template (named as job_name) =====
cat << 'EOF' > ${job_name}.sh
#!/bin/bash -l
#BSUB -P acc_ad-omics
#BSUB -q premium
#BSUB -J JOB_NAME
#BSUB -n 2
#BSUB -R span[hosts=1]
#BSUB -R rusage[mem=30000]
#BSUB -W 42:00
#BSUB -o /WORK_DIR/log/JOB_NAME.%J.out
#BSUB -e /WORK_DIR/log/JOB_NAME.%J.err

export PATH=/sc/arion/projects/ad-omics/haochen/env/R_sc/bin:$PATH:/usr/bin:/bin

sos run "/sc/arion/projects/ad-omics/haochen/rss_qc/2026_Simulation_Sinai/09.Fig2F_3_S5_S6.Finemapping.ipynb" LD_pairwise \
    --resfile `ls SIM_RESULT/*.rds` \
    --cwd WORK_DIR/
EOF
chmod +x ${job_name}.sh
base_sh=${job_name}.sh
work_dir="${job_dir}"

output_script="${job_name}_submit.sh"
sed -e "s|WORK_DIR|${work_dir}|g" \
    -e "s|JOB_NAME|${job_name}|g" \
    -e "s|SOS_FILE|${sos_file}|g" \
    -e "s|SIM_RESULT|${sim_result}|g" \
  "${base_sh}" > "${output_script}"
sb ${output_script}   # uncomment to submit

In [ ]:
# ===== Base config =====
base_dir="/sc/arion/projects/ad-omics/haochen/rss_qc/2026_output/"
job_name="LD_pairwise_EUR"
job_dir="${base_dir}/${job_name}"
sim_result="/sc/arion/projects/ad-omics/haochen/rss_qc/2026_output/Gtex_EUR_simulation/result/h2g_0.1/causal_1/"
# ===== Create structure =====
mkdir -p "${job_dir}"/{code,result,log}
cd "${job_dir}/code"

# ===== Base LSF script template (named as job_name) =====
cat << 'EOF' > ${job_name}.sh
#!/bin/bash -l
#BSUB -P acc_ad-omics
#BSUB -q premium
#BSUB -J JOB_NAME
#BSUB -n 2
#BSUB -R span[hosts=1]
#BSUB -R rusage[mem=30000]
#BSUB -W 42:00
#BSUB -o /WORK_DIR/log/JOB_NAME.%J.out
#BSUB -e /WORK_DIR/log/JOB_NAME.%J.err

export PATH=/sc/arion/projects/ad-omics/haochen/env/R_sc/bin:$PATH:/usr/bin:/bin

sos run "/sc/arion/projects/ad-omics/haochen/rss_qc/2026_Simulation_Sinai/09.Fig2F_3_S5_S6.Finemapping.ipynb" LD_pairwise \
    --resfile `ls SIM_RESULT/*.rds` \
    --cwd WORK_DIR/
EOF
chmod +x ${job_name}.sh
base_sh=${job_name}.sh
work_dir="${job_dir}"

output_script="${job_name}_submit.sh"
sed -e "s|WORK_DIR|${work_dir}|g" \
    -e "s|JOB_NAME|${job_name}|g" \
    -e "s|SOS_FILE|${sos_file}|g" \
    -e "s|SIM_RESULT|${sim_result}|g" \
  "${base_sh}" > "${output_script}"
sb ${output_script}   # uncomment to submit

In [2]:
[finemap_compare]
parameter: folder = path
parameter: cwd = path("output")
parameter: numThreads = 3
parameter: container = ""
input: folder
output: f'{cwd:a}/cs_mix_vs_base_combined.rds', f'{cwd:a}/cs_mix_vs_base_combined.csv'
task: trunk_workers = 1, walltime = walltime, mem = mem, cores = numThreads,
      tags = f'{step_name}_{_output[0]:bn}'
R:  expand = '${ }', stdout = f"{_output[0]:n}.stdout", stderr = f"{_output[0]:n}.stderr", container = container

    suppressMessages(library(parallel))
    folder <- c(${_input:r,})
    files = list.files(folder, full.names = T, recursive = T, pattern = ".rds")

    refs <- c("insample", "ADSP", "UKB", "kg")   # base slot names; _mix appended

    ## credible sets as a canonical set-of-sets: each CS = sorted variant IDs
    get_cs <- function(fit) {
      cs <- tryCatch(fit$sets$cs, error = function(e) NULL)
      if (is.null(cs) || !length(cs)) return(character(0))
      ids <- tryCatch(names(fit$pip), error = function(e) NULL)
      keys <- vapply(cs, function(v) {
        v <- as.integer(v)
        m <- if (!is.null(ids)) sort(ids[v]) else sort(v)
        paste(m, collapse = "|")
      }, character(1))
      keys                                        # one string per credible set
    }

    compare_cs <- function(a, b) {
      shared <- length(intersect(a, b))
      data.frame(
        n_cs      = length(a),
        n_cs_mix  = length(b),
        n_shared  = shared,
        same_all  = (length(a) == length(b)) && (shared == length(a)),
        both_empty= (length(a) == 0 && length(b) == 0)
      )
    }

    process_one <- function(f) {
      tryCatch({
        res <- readRDS(f)$result_list          # <- slots live here
        do.call(rbind, lapply(refs, function(r) {
          a <- get_cs(res[[r]])
          b <- get_cs(res[[paste0(r, "_mix")]])
          data.frame(file = basename(f), reference = r,
                     compare_cs(a, b), row.names = NULL)
        }))
      }, error = function(e) { message("FAILED: ", basename(f), " : ", conditionMessage(e)); NULL })
    }

    parts <- mclapply(files, process_one, mc.cores = ${numThreads})
    ok    <- !vapply(parts, is.null, logical(1))
    dat   <- do.call(rbind, parts[ok])

    saveRDS(dat, ${_output[0]:r})
    write.csv(dat, ${_output[1]:r}, row.names = FALSE)
    cat(sprintf("combined %d/%d regions -> %d rows\n", sum(ok), length(files), nrow(dat)))

ERROR: Error in parse(text = input): <text>:1:1: unexpected '['
1: [
    ^


In [ ]:
work_dir="/sc/arion/projects/ad-omics/haochen/rss_qc/2026_output/Gtex_fine/"
mkdir -p ${work_dir}/summary_cs_similarity
cd ${work_dir}/summary_cs_similarity

# Create the base_script file and write the bash code into it
cat << 'EOF' > base_script
#!/bin/bash -l
#BSUB -P acc_ad-omics
#BSUB -q premium
#BSUB -J finemap_compare
#BSUB -n 3
#BSUB -R span[hosts=1]
#BSUB -R rusage[mem=36000]
#BSUB -W 16:00
#BSUB -o WORK_DIR/summary/finemap_compare.%J.out
#BSUB -e WORK_DIR/summary/finemap_compare.%J.err


export PATH=/sc/arion/projects/ad-omics/haochen/env/R_sc/bin:$PATH:/usr/bin:/bin

sos run "/sc/arion/projects/ad-omics/haochen/rss_qc/2026_Simulation_Sinai/09.Fig2F_3_S5_S6.Finemapping.ipynb" finemap_compare \
    --folder WORK_DIR/result/ \
    --cwd WORK_DIR/summary_cs_similarity


EOF

base_sh="base_script"

output_script="finemap_compare.sh"
cat ${base_sh} | sed "s|WORK_DIR|${work_dir}|g"    > ${output_script}
sb ${output_script}